In [ ]:
# Load some test data
import polars as pl
mpg = pl.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/mpg.csv', infer_schema_length=None)
mpg.head()

mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin,name
f64,i64,f64,str,i64,f64,i64,i64,str
18.0,8,307.0,"""130""",3504,12.0,70,1,"""chevrolet chevelle malibu"""
15.0,8,350.0,"""165""",3693,11.5,70,1,"""buick skylark 320"""
18.0,8,318.0,"""150""",3436,11.0,70,1,"""plymouth satellite"""
16.0,8,304.0,"""150""",3433,12.0,70,1,"""amc rebel sst"""
17.0,8,302.0,"""140""",3449,10.5,70,1,"""ford torino"""


In [ ]:
mpg.describe()

statistic,mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin,name
str,f64,f64,f64,str,f64,f64,f64,f64,str
"""count""",398.0,398.0,398.0,"""398""",398.0,398.0,398.0,398.0,"""398"""
"""null_count""",0.0,0.0,0.0,"""0""",0.0,0.0,0.0,0.0,"""0"""
"""mean""",23.514573,5.454774,193.425879,null,2970.424623,15.56809,76.01005,1.572864,null
"""std""",7.815984,1.701004,104.269838,null,846.841774,2.757689,3.697627,0.802055,null
"""min""",9.0,3.0,68.0,"""100""",1613.0,8.0,70.0,1.0,"""amc ambassador brougham"""
"""25%""",17.5,4.0,104.0,null,2223.0,13.8,73.0,1.0,null
"""50%""",23.0,4.0,151.0,null,2807.0,15.5,76.0,1.0,null
"""75%""",29.0,8.0,262.0,null,3609.0,17.2,79.0,2.0,null
"""max""",46.6,8.0,455.0,"""?""",5140.0,24.8,82.0,3.0,"""vw rabbit custom"""


In [ ]:
# Import the libraries we need
# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow import keras

# Commonly used modules
import numpy as np

# Images, plots, display, and visualization
%pip install lets-plot
from lets_plot import *
LetsPlot.setup_html()
import matplotlib.pyplot as plt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score


In [ ]:
# need to deal with ?s in the data
mpg[32,'horsepower'] = 72
mpg[126,'horsepower'] = 85
mpg[330,'horsepower'] = 64
mpg[354,'horsepower'] = 83
mpg[336,'horsepower'] = 75
mpg[374,'horsepower'] = 90
mpg = mpg.with_columns(pl.col('horsepower').cast(pl.Int64))

In [ ]:
# one hot encode origin
mpg = mpg.with_columns(pl.col('origin').replace_strict({1: 'USA', 2: 'Europe', 3: 'Japan'}))

mpg = mpg.to_dummies(columns=['origin'])


In [ ]:
mpg

mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin_Europe,origin_Japan,origin_USA,name
f64,i64,f64,i64,i64,f64,i64,u8,u8,u8,str
18.0,8,307.0,130,3504,12.0,70,0,0,1,"""chevrolet chevelle malibu"""
15.0,8,350.0,165,3693,11.5,70,0,0,1,"""buick skylark 320"""
18.0,8,318.0,150,3436,11.0,70,0,0,1,"""plymouth satellite"""
16.0,8,304.0,150,3433,12.0,70,0,0,1,"""amc rebel sst"""
17.0,8,302.0,140,3449,10.5,70,0,0,1,"""ford torino"""
…,…,…,…,…,…,…,…,…,…,…
27.0,4,140.0,86,2790,15.6,82,0,0,1,"""ford mustang gl"""
44.0,4,97.0,52,2130,24.6,82,1,0,0,"""vw pickup"""
32.0,4,135.0,84,2295,11.6,82,0,0,1,"""dodge rampage"""


In [ ]:
X = mpg.drop("mpg","name")
y = mpg.select("mpg")

In [ ]:
# look at data, now it is all numeric
X


cylinders,displacement,horsepower,weight,acceleration,year,origin_Europe,origin_Japan,origin_USA
i64,f64,i64,i64,f64,i64,u8,u8,u8
8,307.0,130,3504,12.0,70,0,0,1
8,350.0,165,3693,11.5,70,0,0,1
8,318.0,150,3436,11.0,70,0,0,1
8,304.0,150,3433,12.0,70,0,0,1
8,302.0,140,3449,10.5,70,0,0,1
…,…,…,…,…,…,…,…,…
4,140.0,86,2790,15.6,82,0,0,1
4,97.0,52,2130,24.6,82,1,0,0
4,135.0,84,2295,11.6,82,0,0,1


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
y_train

mpg
f64
15.0
21.0
21.0
20.5
21.0
…
19.0
12.0
21.1


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# fit scaler on training data
norm = MinMaxScaler().fit(X_train)

# transform training data
X_train = norm.transform(X_train)

# transform testing dataabs
X_test = norm.transform(X_test)

In [ ]:
# len(X_train[0])

In [ ]:
# build a neural network using a sequential model
# input layer node size is the same as the training columns
# hidden layers: 128 node layer, 256 node layer, 64 node layer
# a single node output layer
model = Sequential()
model.add(Dense(128, input_dim=len(X_train[0]), activation='relu'))
model.add(Dropout(.5))
model.add(Dense(256, activation='relu'))
model.add(Dense(64, activation='leaky_relu'))

model.add(Dense(1, activation='relu'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 128)            │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,817 (198.50 KB)

 Trainable params: 50,817 (198.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# opt = keras.optimizers.Adam(learning_rate=0.0001)
opt = keras.optimizers.Adam()
model.compile(loss="mean_squared_error", optimizer=opt, metrics=['mse'])

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_mse', patience=30)

history = model.fit(X_train, y_train, epochs=2000, validation_split=.35, batch_size=20, callbacks=[early_stop],shuffle=False)
# history = model.fit(train_features, train_labels, epochs=2000, verbose=0, validation_split = .2, batch_size=tester2,
#                     callbacks=[early_stop, PrintDot()])

hist = pl.DataFrame(history.history)

Epoch 1/2000
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 164ms/step - loss: 602.0081 - mse: 602.0081 - val_loss: 574.7354 - val_mse: 574.7354
Epoch 2/2000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 532.9156 - mse: 532.9156 - val_loss: 474.6080 - val_mse: 474.6080
Epoch 3/2000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 383.8315 - mse: 383.8315 - val_loss: 263.6950 - val_mse: 263.6950
Epoch 4/2000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 175.5519 - mse: 175.5519 - val_loss: 117.9031 - val_mse: 117.9031
Epoch 5/2000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 123.6004 - mse: 123.6004 - val_loss: 89.7916 - val_mse: 89.7916
Epoch 6/2000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 71.1591 - mse: 71.1591 - val_loss: 66.8156 - val_mse: 66.8156
Epoch 7/2000
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 55.8539 - mse: 55.8539 - val_loss: 50.6758 - val_mse: 50.6758
Epoch 8/2000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 47.7789 - mse: 47.7789 - val_loss: 40.6460 - val_mse: 40.6460
Epoch 9/2000


In [ ]:
hist

loss,mse,val_loss,val_mse
f64,f64,f64,f64
602.008057,602.008057,574.735413,574.735413
532.915649,532.915649,474.608032,474.608032
383.831512,383.831512,263.695007,263.695007
175.551926,175.551926,117.903069,117.903069
123.600426,123.600426,89.791573,89.791573
…,…,…,…
10.067956,10.067956,8.758487,8.758487
11.043855,11.043855,8.454241,8.454241
8.417718,8.417718,8.553119,8.553119


In [ ]:
# h = hist
hist = hist.with_row_index()
# h

In [ ]:
hist

index,loss,mse,val_loss,val_mse
u32,f64,f64,f64,f64
0,602.008057,602.008057,574.735413,574.735413
1,532.915649,532.915649,474.608032,474.608032
2,383.831512,383.831512,263.695007,263.695007
3,175.551926,175.551926,117.903069,117.903069
4,123.600426,123.600426,89.791573,89.791573
…,…,…,…,…
177,10.067956,10.067956,8.758487,8.758487
178,11.043855,11.043855,8.454241,8.454241
179,8.417718,8.417718,8.553119,8.553119


In [ ]:
mse_hist = hist.drop(['loss', 'val_loss']).unpivot(index='index', variable_name='metric', value_name='mse').with_columns(pl.col('metric').replace({'mse': 'training error', 'val_mse': 'validation error'}))
mse_hist

index,metric,mse
u32,str,f64
0,"""training error""",602.008057
1,"""training error""",532.915649
2,"""training error""",383.831512
3,"""training error""",175.551926
4,"""training error""",123.600426
…,…,…
177,"""validation error""",8.758487
178,"""validation error""",8.454241
179,"""validation error""",8.553119


In [ ]:
(
    ggplot(data=mse_hist, mapping=aes(x='index', y='mse', color='metric'))
    + geom_line()
    + labs(x='Epoch', y='Mean Square Error')
    + theme_minimal()
    + theme(legend_title=element_blank())
)

In [ ]:
predictions = np.round(model.predict(X_test),1)

1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step


In [ ]:
predictions

array([[32.6],
       [29. ],
       [20.5],
       [15.1],
       [13.4],
       [24.7],
       [26.6],
       [12.6],
       [16.2],
       [19.7],
       [14. ],
       [36.9],
       [24.2],
       [14.1],
       [22.3],
       [12.4],
       [29.3],
       [20.5],
       [14.8],
       [37.1],
       [20.9],
       [19.4],
       [24.9],
       [27.7],
       [15.5],
       [41.1],
       [22.4],
       [22.6],
       [18.7],
       [12.8],
       [27.5],
       [36.5],
       [17.1],
       [23.4],
       [34.8],
       [13.2],
       [21.7],
       [17.9],
       [13.7],
       [23.2],
       [23.6],
       [25.9],
       [19.5],
       [11.8],
       [22.5],
       [33.6],
       [23.7],
       [20.8],
       [22.3],
       [25.8],
       [21.5],
       [33.8],
       [32.2],
       [12.8],
       [25.6],
       [12.9],
       [17. ],
       [28.5],
       [22.3],
       [18.2],
       [14.4],
       [31.5],
       [22. ],
       [19.3],
       [19.3],
       [23.3],
       [22

In [ ]:

result = root_mean_squared_error(y_test, predictions)
result

2.3708648681640625

In [ ]:

r2 = r2_score(y_test,predictions)
r2

0.9020584225654602

In [ ]:
pred = pl.DataFrame(predictions,schema=['predictions'])
y_test.columns = ['actual']
pred = pl.concat([pred, y_test], how='horizontal').with_columns(difference = pl.col('actual') - pl.col('predictions'))
display(pred)

predictions,actual,difference
f32,f64,f64
32.599998,33.0,0.400002
29.0,28.0,-1.0
20.5,19.0,-1.5
15.1,13.0,-2.1
13.4,14.0,0.6
…,…,…
20.9,20.0,-0.9
23.4,26.0,2.6
19.700001,20.6,0.899999


In [ ]:
(
    ggplot(data=pred, mapping=aes(x='actual', y='predictions'))
    + geom_point(color='blue')
    + geom_smooth(method='lm', color='red', na_rm=True, se=False)
    + scale_x_continuous(limits=(5, 50))
    + scale_y_continuous(limits=(5, 50))
    + labs(x='Actual MPG', y='Predicted MPG')
    + theme_minimal()
)